# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [9]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [10]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

In [5]:
# import uuid4
import os
import getpass
os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - rithy"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [11]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [12]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [13]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [14]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [15]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [16]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [17]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [18]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issues with loans, based on the complaints in the provided data, seem to involve problems with how loans are handled and managed. Specifically, recurring issues include:\n\n- Errors in loan balances and incorrect information on credit reports.\n- Misapplication of payments, often resulting in unintended interest accumulation.\n- Problems with loan transfers and transfers occurring without proper notification or consent.\n- Troubles with repayment plans, including difficulty applying extra funds to principal or paying off smaller loans faster.\n- Discrepancies and errors related to loan status, such as wrongful delinquencies or incorrect reporting of account status.\n- Issues with bad or incorrect information provided about loans, including confusing or inconsistent loan balances and interest calculations.\n- Challenges with loan forgiveness, discharge, or settlement processes.\n\nOverall, these complaints suggest that a major and most common issue is the mishandling an

In [19]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, it appears that some complaints did not get handled in a timely manner. For example, the complaint from 03/28/25 submitted to MOHELA was marked "Timely response? No," indicating it was not addressed promptly. Similarly, the complaints from 04/24/25 to Maximus Federal Services and Aidvantage mentioned delays or failures to respond within expected timeframes, with one complaint specifically noting a delay of over 18 months with no resolution.\n\nTherefore, yes, some complaints were not handled in a timely manner.'

In [20]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors highlighted in the complaints:\n\n1. **Lack of Clear Communication and Notification:** Several complainants reported that they were not adequately informed about when repayment was to begin, the transfer of their loans between servicers, or changes in their payment requirements. For example, some loans transitioned to different servicers without notice, leading to missed payments and credit penalties.\n\n2. **Mismanagement and Errors by Loan Servicers:** Complaints indicate that servicers often failed to provide correct information, failed to notify borrowers of their delinquency, or misprocessed payments. Examples include being reported as delinquent without prior notice, being unable to access accurate account information, and being misled about repayment statuses.\n\n3. **Compounding Interest and Financial Hardship:** Many borrowers found that due to interest accumulating during deferment or forbearance

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [26]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [27]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [87]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to "Dealing with your lender or servicer." Specific sub-issues include disputes over fees, difficulties applying payments correctly, and receiving bad or confusing information about loan balances or terms. Many complaints involve feeling that lenders or servicers are untrustworthy, providing incorrect or misleading information, and issues with repayment processes that seem to favor the lender\'s interests over the borrower\'s.'

In [88]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, all of them indicate that each organization responded in a timely manner. Specifically, the complaints from 04/26/25, 04/01/25, 04/24/25, and 05/08/25 all include the response status "Timely response? Yes." Therefore, there is no evidence in the provided data that any complaints were not handled in a timely manner.'

In [89]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues with the management and communication of their loan servicing, problems with payment plans, and complications arising from the transfer of their loans between different servicers. Specific issues reported include being steered into incorrect payment plans, lack of response from loan servicers when requesting forbearance or deferment, automatic payments being discontinued without proper notification, and billing errors such as reversed payments or past dues that negatively impacted credit scores. Additionally, some borrowers were not properly informed or did not receive communications about changes in their loan status or transfer to new servicers, leading to unintentional missed payments and financial difficulties.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer:
BM25 would excel at exact keyword matching tasks. So for instance if I were to ask "What is the complaint ID for the issue about 'incorrect interest rate'?", it would be able to find that exact phrase in a user's complaint and match it perfectly. The only extra step here would be to find the complaint ID. Embeddings are much better for questions that require semantic understanding of contexts and data.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [36]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [37]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [38]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to dealing with lenders or servicers, such as errors in loan balances, misapplied payments, wrongful denials of payment plans, and mishandling of loan information. Many complaints involve receiving incorrect or inconsistent loan information, unauthorized transfers, privacy violations, and difficulties in communication or resolution.'

In [93]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, there are complaints that did not get handled in a timely manner. For example, the complaint regarding the student loan issue with Maximus Federal Services has been open for nearly 18 months without resolution, despite the consumer requesting updates and a resolution. Additionally, the complaint from EdFinancial Services about payments not appearing on the account was ongoing over 2-3 weeks without resolution.'

In [94]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons based on the context provided:\n\n1. **Lack of Awareness and Information**: Many borrowers were unaware that they needed to repay student loans or were not properly informed about the repayment obligations by financial aid officers.\n\n2. **Administrative Issues and Communication Failures**: Borrowers experienced issues such as being unaware of loan transfers, not receiving notifications about repayment start dates, and difficulty accessing or understanding loan information online.\n\n3. **Financial Hardship and Unmanageable Payments**: Many borrowers found their payments to be unaffordable given their financial circumstances, especially when interest continued to accrue during deferment or forbearance, leading to increased balances over time.\n\n4. **Interest Accumulation During Deferment or Forbearance**: Borrowers reported that interest kept accumulating even when payments were paused, which increased their total debt and ma

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [39]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [41]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [97]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, it appears that the most common issues with student loans include:\n\n- Problems with how payments are being handled, such as difficulty applying extra funds to the principal, trouble with payment plans, or late payments without proper notice.\n- Discrepancies or errors in loan balances, interest calculations, or account status reporting.\n- Poor communication or lack of transparency from servicers regarding loan terms, fees, or account updates.\n- Unauthorized or improper transfers of loans and mishandling of loan information.\n- Issues related to loan forgiveness, discharge, or discharge process mismanagement.\n- Inaccurate reporting to credit bureaus and improper collection practices.\n- Unanswered or conflicting information from loan servicers.\n\nThe most recurring theme seems to be **"Trouble with how payments are being handled" or issues related to payments, account errors, and mismanagement of loan information.**\n\nIf I had to identify one mo

In [98]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints were not handled in a timely manner. Specifically, at least two complaints indicated that responses or resolutions were delayed:\n\n1. Complaint with ID 12739706 (Mohela) received on 04/01/25 was marked as "Not timely," indicating a delay beyond the expected response time.\n\n2. Complaint with ID 12709087 (MOHELA) received on 03/28/25 was also marked as "No," meaning it was not handled within the expected timeframe.\n\nAdditionally, there are multiple instances where consumers reported waiting over the expected response time, such as waiting several days or weeks for a callback or resolution, or noticing that no response was received despite promises.\n\nTherefore, yes, some complaints did not get handled in a timely manner.'

In [99]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to issues such as miscommunication, errors, or misconduct by student loan servicers. Specific reasons include receiving bad information about their loans, being placed into forbearance without proper notice, misapplied or unapplied payments, and being steered into long-term forbearances or consolidation practices that increased their debt. In some cases, borrowers were not informed of available repayment options like income-driven plans or loan rehabilitation, which could have helped them manage or reduce their debt. Additionally, systemic failures like improper account management, inaccurate reporting, and violation of regulations contributed to difficulties in repayment, leading to missed payments and negative impacts on credit scores.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer:
It increases the chances of retrieving all of the relevant documents, even if different wording/phrasing is used than that of the original query. This reduces the risk of missing relevant info due to varying verbage.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [42]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [43]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [44]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [45]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [46]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [105]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"The most common issue with loans, based on the provided complaints, appears to be problems related to federal student loan servicing, such as errors in loan balances, misapplied payments, wrongful denials of payment plans, and issues with loan reporting and information accuracy. Many complaints highlight systemic breakdowns, errors, and misconduct by loan servicers that negatively impact borrowers' credit reports and financial situations."

In [106]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, based on the provided information, several complaints did not get handled in a timely manner. For example:\n\n- The complaint against MOHELA received on 03/28/25 was marked as "Timely response?": No, indicating it was not handled promptly.\n- The complaint against MOHELA received on 04/11/25 also was "Timely response?": No.\n- Even though the complaint regarding Aidvantage received on 04/11/25 was marked as "Yes," other complaints, such as the one against MOHELA on 04/11/25 and the dispute settlement sent to credit bureaus on 04/27/25, indicate delays or lack of prompt handling.\n\nOverall, several complaints explicitly indicate delays or failure to respond in a timely manner.'

In [107]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to various difficulties and issues such as financial hardship, mismanagement, lack of proper information, and issues with loan servicers. For example, some borrowers experienced severe financial hardship after graduation, making it difficult to make consistent payments. Others faced problems with loan servicing practices, like being reported as delinquent while still resolving issues, or not being properly notified about payment obligations. Additionally, some borrowers took on loans based on misleading information about the value of their education or institutional stability, which impacted their ability to repay. Overall, these challenges led to struggles in repaying their loans successfully.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [47]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [48]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [110]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided data, the most common issues with student loans tend to involve:\n\n- Dealing with lenders or servicers, including mismanagement, misapplied payments, incorrect information, and problems with payment handling.\n- Errors in loan balances and interest calculations.\n- Receiving bad or false information about loan terms, interest accrual, or account status.\n- Problems with loan transfers without proper notice.\n- Disputes over fees charged, fees discrepancies, or incorrect account classifications.\n- Difficulties with loan consolidation, improper ending of deferments, or incorrect loan classification.\n- Issues related to credit reporting errors, late payments reported inaccurately, or damaging credit scores.\n- Mishandling of applications for forgiveness, cancellation, or discharge, especially related to income-driven plans or Public Service Loan Forgiveness.\n- Unauthorized access or privacy violations.\n\nWhile multiple complaints show varied problems, a recurri

In [111]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, there are multiple instances where complaints were not handled in a timely manner. Many complaints explicitly state that responses were delayed beyond acceptable timeframes, such as delays of several days or weeks, or complaints where the company failed to respond at all despite repeated follow-ups. For example:\n\n- The complaint at row 441 indicates a response was not timely, with the complaint marked "No" under "Timely response."\n- Multiple complaints (such as row 12935889 and row 13365901) show delays of over 30 days or no response at all, despite the complainants\' repeated follow-ups.\n- Some complaints mention that the companies failed to respond or investigate within the required timeframes, leading to unresolved issues and exacerbated harm.\n\nTherefore, it can be concluded that some complaints did not get handled in a timely manner.'

In [112]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. **Limited or Ineffective Payment Options**: Many borrowers were only offered options like forbearance or deferment, which allowed interest to continue accumulating, making repayment more difficult over time.\n\n2. **Accumulation of Interest and Loan Balances**: Due to interest capitalizing during forbearance and lack of clear guidance, many saw their balances grow substantially, sometimes doubling or tripling, which extended repayment periods and increased total debt.\n\n3. **Lack of Transparent Communication**: Borrowers often were not properly notified about changes in loan servicing, transfer of loans between companies, or the start of repayment periods, leading to missed payments and negative credit impacts.\n\n4. **Mismanagement and Misleading Practices by Servicers**: Some borrowers experienced long-term forbearance steering, incorrect reporting to credit bureaus, or failed to receive proper documentatio

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [49]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [50]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [51]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [52]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [53]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [118]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided context, a common issue with loans, particularly student loans, appears to be problems related to servicing and communication. Specific frequent issues include:\n\n- Struggling to repay the loan, often due to disputes about payment amounts or inability to get clear information.\n- Problems with loan reporting, such as incorrect or unauthorized reporting of account status or debt, leading to credit score impacts.\n- Difficulties with loan servicing entities, like failure to process applications, delays, or inconsistent information about loan status or issuer.\n- Allegations of improper use or mishandling of personal data, sometimes resulting in breach of federal privacy laws.\n- Issues with the handling of repayment plans, including incorrect billing, failure to process recertifications, or unclear communication about payment obligations.\n\nWhile the specific most common issue can vary, the recurring theme across complaints is **problems with loan servicing, comm

In [119]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, there are several complaints indicating that issues were not handled in a timely manner. \n\nSpecifically, the complaint about Nelnet (ID: 13331376 received on 05/04/25) mentions that despite multiple letters sent via Certified Mail acknowledging receipt and detailing misconduct, Nelnet never responded to the complaint or provided answers, indicating a lack of timely handling. The company responded with "Closed with explanation," but the narrative suggests delays and lack of response.\n\nOther complaints also refer to ongoing issues, such as incorrect payment processing and disputes that have not been resolved promptly, although the responses in those cases were marked as "Yes" for being timely, but the narratives show ongoing unresolved issues.\n\nIn summary, yes, there are complaints that were not handled in a timely manner, particularly the case involving Nelnet where the consumer waited without response despite clear attempts to communicate.'

In [120]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"Based on the provided information, people failed to pay back their loans due to a variety of reasons, including:\n\n1. Lack of transparency and communication issues with lenders or servicers, leading to confusion and stress (e.g., receiving bad information about their loans or difficulties accessing their accounts).\n2. Problems with the handling of payment plans, such as re-amortization failures after the end of COVID-19 forbearance, resulting in unexpectedly increased payments.\n3. Disputes over the accuracy of reported account status, like being wrongly classified as delinquent or in default, which impacts credit scores and borrowing ability.\n4. Delays or errors in payment processing, including payments not being recognized or recorded correctly.\n5. Legal and administrative issues, such as the transfer of loans between servicers and missing payments, or challenges related to loan forgiveness and discharge claims.\n6. Alleged misconduct or illegal practices by loan servicers, incl

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer:
Performance would definitely suffer for semantic chunking, due to low semantic variation. This would also result in over chunking, creating too many tiny chunks since no clear semantic boundaries can be found. I would adjust the algorithm by increasing the minimum chunk size and using different thresholding other than percentile. Pre-processing to remove exact duplicates before chunking would also remediate many of the problems with FAQ style sentences.

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### ✅ Answer:
 bm25: 
 {'context_recall': 0.0000, 'faithfulness': 0.0930, 'factual_correctness': 0.3892, 'answer_relevancy': 0.9504, 'context_entity_recall': 0.0362, 'noise_sensitivity_relevant': 0.0000}

 naive: 
 {'context_recall': 0.0000, 'faithfulness': 0.2740, 'factual_correctness': 0.3858, 'answer_relevancy': 0.7925, 'context_entity_recall': 0.0605, 'noise_sensitivity_relevant': 0.0000}

 compression: 
 {'context_recall': 0.0000, 'faithfulness': 0.1238, 'factual_correctness': 0.4467, 'answer_relevancy': 0.8663, 'context_entity_recall': 0.0572, 'noise_sensitivity_relevant': 0.0000}

 multi_query: 
 {'context_recall': 0.0375, 'faithfulness': 0.3711, 'factual_correctness': 0.4033, 'answer_relevancy': 0.8707, 'context_entity_recall': 0.1056, 'noise_sensitivity_relevant': 0.0000}

 parent_document: 
 {'context_recall': 0.0000, 'faithfulness': 0.1352, 'factual_correctness': 0.4900, 'answer_relevancy': 0.7913, 'context_entity_recall': 0.0481, 'noise_sensitivity_relevant': 0.0048}
 
 ensemble: 
 {'context_recall': 0.0833, 'faithfulness': 0.1773, 'factual_correctness': 0.4250, 'answer_relevancy': 0.9521, 'context_entity_recall': 0.0425, 'noise_sensitivity_relevant': 0.0000}
 
For this particular dataset, the Ensemble Retriever stands out as the best-performing method. By combining the strengths of multiple retrieval strategies, it achieves the highest context recall and answer relevancy among all tested retrievers, ensuring that responses are both accurate and highly relevant to user queries. While some individual retrievers excel in specific metrics, the ensemble approach provides a balanced performance across all key areas, including factual correctness and low noise sensitivity. This makes it especially effective for complex, real-world data like loan complaints, where capturing diverse aspects of the context and delivering precise, relevant answers is crucial.


##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [21]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from ragas.testset import TestsetGenerator

path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

dataset.to_pandas()



Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/29 [00:00<?, ?it/s]

Property 'summary' already exists in node '34d9e7'. Skipping!
Property 'summary' already exists in node 'eacd94'. Skipping!
Property 'summary' already exists in node '97ff9c'. Skipping!
Property 'summary' already exists in node 'f8df6d'. Skipping!
Property 'summary' already exists in node '717f91'. Skipping!
Property 'summary' already exists in node 'e6894e'. Skipping!
Property 'summary' already exists in node '277716'. Skipping!
Property 'summary' already exists in node '149681'. Skipping!
Property 'summary' already exists in node '13fc98'. Skipping!
Property 'summary' already exists in node '416eb1'. Skipping!
Property 'summary' already exists in node '12e71b'. Skipping!
Property 'summary' already exists in node '157923'. Skipping!


...

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,Can dependent undergraduates receive Direct Un...,"[information, see the discussion under <Direct...",If you verify that the parents of a dependent ...,single_hop_specifc_query_synthesizer
1,Whoo canm be consdiered a parent on the FAFSAA...,[Definition of <Parent= for Direct PLUS Loan P...,"For Direct PLUS Loan purposes, a parent on the...",single_hop_specifc_query_synthesizer
2,What is the purpose of the Federal Direct PLUS...,[If your school participates in the Direct PLU...,The Federal Direct PLUS Loan Request for Suppl...,single_hop_specifc_query_synthesizer
3,"According to 34 CFR 668.164(j)(3)(iii), when c...",[Direct Loan Eligibility After an Enrollment S...,A school may make a late disbursement of a Dir...,single_hop_specifc_query_synthesizer
4,"According to federal regulations, who is eligi...","[<1-hop>\n\ninformation, see the discussion un...",A Direct PLUS Loan can be borrowed on behalf o...,multi_hop_abstract_query_synthesizer
5,"According to federal regulations, who is eligi...","[<1-hop>\n\ninformation, see the discussion un...",A Direct PLUS Loan can be borrowed on behalf o...,multi_hop_abstract_query_synthesizer
6,If a parent with an adverse credit history app...,"[<1-hop>\n\ninformation, see the discussion un...",If a parent applying for a Direct PLUS Loan ha...,multi_hop_abstract_query_synthesizer
7,Who is eligible to borrow Direct PLUS Loans on...,"[<1-hop>\n\ninformation, see the discussion un...",Direct PLUS Loans can be borrowed on behalf of...,multi_hop_abstract_query_synthesizer
8,What are the requirements for Direct Loan exit...,[<1-hop>\n\nIf your school participates in the...,Direct Loan exit counseling must provide borro...,multi_hop_specific_query_synthesizer
9,if a student parent cant get a direct plus loa...,[<1-hop>\n\nIf your school participates in the...,if a student’s parent cant get a direct plus l...,multi_hop_specific_query_synthesizer


In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks.tracers.langchain import LangChainTracer
from langchain.callbacks.manager import CallbackManager


def make_chat_model(retriever_name: str = None, model_name: str = "gpt-4o") -> ChatOpenAI:
    project_name=f"ragas-rkv-eval-{retriever_name}"
    print(project_name)
    if retriever_name:
        tracer = LangChainTracer(
            project_name=project_name,
            tags=[retriever_name, "retriever", model_name]
        )
        callback_manager = CallbackManager([tracer])
        return ChatOpenAI(
            model=model_name,
            temperature=0,
            max_tokens=8192,
            callback_manager=callback_manager
        )
metric_map = {}

In [ ]:
import copy
from ragas import EvaluationDataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
# from langsmith.evaluation import LangChainStringEvaluator, evaluate
from ragas import RunConfig
import time

retrievers = {
    "naive": naive_retrieval_chain,
    "bm25": bm25_retrieval_chain,
    "compression": contextual_compression_retrieval_chain,
    "multi_query": multi_query_retrieval_chain,
    "parent_document": parent_document_retrieval_chain,
    "ensemble": ensemble_retrieval_chain
}

for retriever_name in retrievers:
    retriever_chain = retrievers[retriever_name]
    for test_row in dataset:
        response = retriever_chain.invoke({"question" : test_row.eval_sample.user_input})
        test_row.eval_sample.response = response["response"].content
        test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
        time.sleep(7)
        
    evaluator_llm = LangchainLLMWrapper(make_chat_model(retriever_name=retriever_name,model_name="gpt-4.1-mini"))

    evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())
    custom_run_config = RunConfig(timeout=360)

    result = evaluate(
        dataset=evaluation_dataset,
        metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
        llm=evaluator_llm,
        run_config=custom_run_config
    )
    
    metric_map[retriever_name] = result
        


ragas-rkv-eval-naive


/tmp/ipykernel_18023/3423380389.py:15: DeprecationWarning: callback_manager is deprecated. Please use callbacks instead.
  return ChatOpenAI(


Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[10]: LLMDidNotFinishException(The LLM generation was not completed. Please increase try increasing the max_tokens and try again.)
Exception raised in Job[28]: LLMDidNotFinishException(The LLM generation was not completed. Please increase try increasing the max_tokens and try again.)
Exception raised in Job[64]: LLMDidNotFinishException(The LLM generation was not completed. Please increase try increasing the max_tokens and try again.)
Exception raised in Job[29]: TimeoutError()
Exception raised in Job[35]: TimeoutError()
Exception raised in Job[41]: TimeoutError()
Exception raised in Job[53]: TimeoutError()
Exception raised in Job[59]: TimeoutError()
Exception raised in Job[65]: TimeoutError()
Exception raised in Job[71]: TimeoutError()


ragas-rkv-eval-compression


...

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

ragas-rkv-eval-ensemble


/tmp/ipykernel_18023/3423380389.py:15: DeprecationWarning: callback_manager is deprecated. Please use callbacks instead.
  return ChatOpenAI(


Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[10]: LLMDidNotFinishException(The LLM generation was not completed. Please increase try increasing the max_tokens and try again.)
Exception raised in Job[16]: LLMDidNotFinishException(The LLM generation was not completed. Please increase try increasing the max_tokens and try again.)
Exception raised in Job[40]: LLMDidNotFinishException(The LLM generation was not completed. Please increase try increasing the max_tokens and try again.)
Exception raised in Job[52]: LLMDidNotFinishException(The LLM generation was not completed. Please increase try increasing the max_tokens and try again.)
Exception raised in Job[58]: LLMDidNotFinishException(The LLM generation was not completed. Please increase try increasing the max_tokens and try again.)
Exception raised in Job[64]: LLMDidNotFinishException(The LLM generation was not completed. Please increase try increasing the max_tokens and try again.)
Exception raised in Job[70]: LLMDidNotFinishException(The LLM generation was

In [ ]:
for key in metric_map:
    print(key + ': ' )
    print(metric_map[key])    

# bm25: 
# {'context_recall': 0.0000, 'faithfulness': 0.0930, 'factual_correctness': 0.3892, 'answer_relevancy': 0.9504, 'context_entity_recall': 0.0362, 'noise_sensitivity_relevant': 0.0000}
# naive: 
# {'context_recall': 0.0000, 'faithfulness': 0.2740, 'factual_correctness': 0.3858, 'answer_relevancy': 0.7925, 'context_entity_recall': 0.0605, 'noise_sensitivity_relevant': 0.0000}
# compression: 
# {'context_recall': 0.0000, 'faithfulness': 0.1238, 'factual_correctness': 0.4467, 'answer_relevancy': 0.8663, 'context_entity_recall': 0.0572, 'noise_sensitivity_relevant': 0.0000}
# multi_query: 
# {'context_recall': 0.0375, 'faithfulness': 0.3711, 'factual_correctness': 0.4033, 'answer_relevancy': 0.8707, 'context_entity_recall': 0.1056, 'noise_sensitivity_relevant': 0.0000}
# parent_document: 
# {'context_recall': 0.0000, 'faithfulness': 0.1352, 'factual_correctness': 0.4900, 'answer_relevancy': 0.7913, 'context_entity_recall': 0.0481, 'noise_sensitivity_relevant': 0.0048}
# ensemble: 
# {'context_recall': 0.0833, 'faithfulness': 0.1773, 'factual_correctness': 0.4250, 'answer_relevancy': 0.9521, 'context_entity_recall': 0.0425, 'noise_sensitivity_relevant': 0.0000}



bm25: 
{'context_recall': 0.0000, 'faithfulness': 0.0930, 'factual_correctness': 0.3892, 'answer_relevancy': 0.9504, 'context_entity_recall': 0.0362, 'noise_sensitivity_relevant': 0.0000}
naive: 
{'context_recall': 0.0000, 'faithfulness': 0.2740, 'factual_correctness': 0.3858, 'answer_relevancy': 0.7925, 'context_entity_recall': 0.0605, 'noise_sensitivity_relevant': 0.0000}
compression: 
{'context_recall': 0.0000, 'faithfulness': 0.1238, 'factual_correctness': 0.4467, 'answer_relevancy': 0.8663, 'context_entity_recall': 0.0572, 'noise_sensitivity_relevant': 0.0000}
multi_query: 
{'context_recall': 0.0375, 'faithfulness': 0.3711, 'factual_correctness': 0.4033, 'answer_relevancy': 0.8707, 'context_entity_recall': 0.1056, 'noise_sensitivity_relevant': 0.0000}
parent_document: 
{'context_recall': 0.0000, 'faithfulness': 0.1352, 'factual_correctness': 0.4900, 'answer_relevancy': 0.7913, 'context_entity_recall': 0.0481, 'noise_sensitivity_relevant': 0.0048}
ensemble: 
{'context_recall': 0.08